# Parallel Subgraph Execution in LangGraph
## Concurrent Branches — Fan Out and Reduce in LangGraph
⏱ ~12 min

**Parallel subgraph execution** is the key pattern for building high-throughput LangGraph pipelines. Instead of processing items one at a time, the **Send API** fans work out to N independent workers that run simultaneously — wall-clock time equals the *slowest single worker*, not the sum of all workers. After all workers finish, a single fan-in node merges the results.

By the end of this session you will understand *why* parallelism matters, *how* the Send API dispatches work, *how* `Annotated[list, operator.add]` automatically merges parallel outputs, and *how* to build production-ready map-reduce graphs from scratch.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Sequential vs parallel execution, the map-reduce pattern |
| 2 | **Send API Mechanics** — How `Send("node", state)` works |
| 3 | **State Merging** — `Annotated[list, operator.add]` fan-in |
| 4 | **Building the Graph** — distribute → [N×workers] → aggregate |
| 5 | **Running and Observing** — Invoke, inspect, measure the speedup |
| 6 | **Variations** — Heterogeneous workers, index preservation, conditional routing |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with requirements already installed, or run on Colab)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> LangGraph Send API how-to: https://langchain-ai.github.io/langgraph/how-tos/map-reduce/  
> Dean, J. & Ghemawat, S. (2004). *MapReduce: Simplified Data Processing on Large Clusters.* OSDI 2004.  
> LangGraph conceptual guide — Send: https://langchain-ai.github.io/langgraph/concepts/low_level/#send  
> Companion examples: `26-rag-fusion` (Send for parallel retrieval), `6-multi-agent-graph` (multiple agents in one graph)

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import operator
import os
import time
import threading
from typing import Annotated, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Concepts: Sequential vs Parallel Execution

---

### The Problem with Sequential Processing

Most LLM pipelines process items one at a time. If you have 8 documents and each LLM call takes 2 seconds, sequential processing takes **16 seconds**. Most of that time the CPU is idle, waiting for the previous API call to return.

Parallel execution dispatches all 8 calls at once. Total time drops to roughly **2 seconds** — the slowest single worker.

```
Sequential (16 s total):
  doc1 ──2s──►
               doc2 ──2s──►
                             doc3 ──2s──►  ...

Parallel (2 s total):
  doc1 ──2s──►
  doc2 ──2s──►
  doc3 ──2s──►
  ...          (all finish ~simultaneously)
```

---

### The Map-Reduce Pattern

Map-reduce is the classic parallel computation pattern:

| Phase | What happens | LangGraph component |
|-------|-------------|--------------------|
| **Map (fan-out)** | Distribute N items to N independent workers | `distribute()` returning `[Send(...), ...]` |
| **Work** | Each worker processes its item independently | `summarize()` node running in parallel |
| **Reduce (fan-in)** | Collect and merge all worker outputs | `Annotated[list, operator.add]` + `aggregate()` |

---

### LangGraph Parallel Execution Architecture

```
INPUT
  {"documents": [doc_A, doc_B, doc_C, doc_D]}
        │
        ▼
  ┌─────────────┐
  │  distribute  │  conditional edge from START
  │  (fan-out)   │  returns [Send("summarize", {doc: A}),
  └──────┬───────┘           Send("summarize", {doc: B}),
         │                   Send("summarize", {doc: C}),
         │                   Send("summarize", {doc: D})]
         │
         ├──────────────────────────────────
         │                │                │
  ┌──────▼──────┐  ┌──────▼──────┐  ┌──────▼──────┐
  │  summarize  │  │  summarize  │  │  summarize  │  ...
  │  (doc: A)   │  │  (doc: B)   │  │  (doc: C)   │
  │ → sum_A     │  │ → sum_B     │  │ → sum_C     │
  └──────┬──────┘  └──────┬──────┘  └──────┬──────┘
         │                │                 │
         └────────────────┴─────────────────┘
                          │
              Annotated[list, operator.add]
              merges all summaries automatically
                          │
                          ▼
                  ┌───────────────┐
                  │   aggregate   │  called ONCE after ALL workers done
                  │  (fan-in)     │  receives [sum_A, sum_B, sum_C, sum_D]
                  └───────┬───────┘
                          │
                         END
                  {"final_summary": "..."}
```

---

### Vocabulary

| Term | Definition |
|------|------------|
| **Send API** | LangGraph mechanism to dispatch a node call with a custom state payload |
| **Fan-out** | Splitting one input into N parallel branches |
| **Fan-in** | Merging N parallel results back into one state |
| **Conditional edge** | An edge whose target is determined at runtime by a function's return value |
| **`Annotated[list, operator.add]`** | A TypedDict field that LangGraph merges by appending rather than replacing |
| **Worker node** | A regular node that receives a per-item state slice from `Send` |
| **Aggregate node** | The fan-in node that runs once after all workers complete |
| **Wall-clock time** | Actual elapsed time (parallel = max worker time, not sum) |

---

## Part 2 — Send API Mechanics

---

### What `Send` Actually Does

`Send("node_name", state_dict)` is a message that tells LangGraph: *"call `node_name` with this exact state dict, right now, in parallel with any other Sends returned from this function."*

The function that returns `[Send(...), ...]` is wired as a **conditional edge** from a source node (or `START`). LangGraph inspects the return value:

- If it returns a string → go to that named node
- If it returns a list of strings → go to all those nodes
- If it returns a list of `Send` objects → dispatch each as an independent parallel call

```python
# The distribute function is a conditional edge, not a node.
# It returns Send objects — one per document.
def distribute(state: State):
    return [
        Send("summarize", {"doc": doc, "summaries": []})
        for doc in state["documents"]
    ]

# Wire it to START — LangGraph runs distribute, then fans out
graph.add_conditional_edges(START, distribute, ["summarize"])
# Third argument lists ALL allowed target nodes (safety declaration)
```

---

### Per-Worker State Isolation

Each `Send` call gets its **own copy** of the state dict you pass. Workers do not share mutable state — they cannot accidentally interfere with each other.

```
Send("summarize", {"doc": "LangGraph is...", "summaries": []})
                   │
                   └─► worker sees: state["doc"] == "LangGraph is..."
                                    state["summaries"] == []
                                    state["documents"] is NOT present
                                    (only what you put in the Send dict)
```

This is different from a normal edge where the full graph state is passed along.

In [ ]:
# ─── 2-A: Inspect Send objects before wiring ──────────────────────────────────
# You can construct and inspect Send objects without running a graph.

sample_docs = [
    "LangGraph 0.6 introduced the Send API for true parallel fan-out.",
    "ReAct agents interleave reasoning steps with tool calls.",
    "RAG pipelines combine a dense retriever with a generative LLM.",
    "Human-in-the-loop checkpointing pauses graphs for human review.",
]


def preview_distribute(documents: list[str]) -> list[Send]:
    """Shows what distribute() returns before wiring into a graph."""
    return [
        Send("summarize", {"doc": doc, "summaries": []})
        for doc in documents
    ]


sends = preview_distribute(sample_docs)

print(f"distribute() returns {len(sends)} Send objects:\n")
for i, send in enumerate(sends):
    print(f"  Send [{i}]")
    print(f"    target node : {send.node}")
    print(f"    state['doc']: {send.arg['doc'][:60]}...")
    print(f"    state keys  : {list(send.arg.keys())}")
    print()

print("All Sends dispatched simultaneously by LangGraph — N workers run in parallel.")

In [ ]:
# ─── 2-B: Sequential baseline — measure how long N serial calls take ──────────
# Simulated timing (no real LLM) to show the sequential vs parallel contrast
# before we build the real graph in Part 4.

SIMULATED_LATENCY_S = 0.4  # pretend each worker takes 0.4 s


def fake_worker(doc: str) -> str:
    """Simulates a slow LLM call."""
    time.sleep(SIMULATED_LATENCY_S)
    return f"[summary of: {doc[:40]}]"


# Sequential
t0 = time.perf_counter()
sequential_results = [fake_worker(doc) for doc in sample_docs]
sequential_time = time.perf_counter() - t0

# Parallel (threads — same model LangGraph uses internally)
parallel_results: list = [None] * len(sample_docs)


def threaded_worker(i: int, doc: str):
    parallel_results[i] = fake_worker(doc)


t0 = time.perf_counter()
threads = [
    threading.Thread(target=threaded_worker, args=(i, doc))
    for i, doc in enumerate(sample_docs)
]
for t in threads:
    t.start()
for t in threads:
    t.join()
parallel_time = time.perf_counter() - t0

print(f"Documents       : {len(sample_docs)}")
print(f"Latency/worker  : {SIMULATED_LATENCY_S:.1f}s")
print()
print(f"Sequential time : {sequential_time:.2f}s  (≈ N × latency)")
print(f"Parallel time   : {parallel_time:.2f}s  (≈ 1 × latency)")
print(f"Speedup         : {sequential_time / parallel_time:.1f}x")
print()
print("LangGraph's Send API achieves the same speedup automatically for LLM calls.")

---

## Part 3 — State Merging with `Annotated[list, operator.add]`

---

### The Merge Problem

When N workers each produce a result, LangGraph needs to collect them into a single state field. The default behavior for TypedDict fields is **replacement**: the last writer wins. That would lose all but one worker's output.

The solution is `Annotated[list, operator.add]`. This tells LangGraph: *"instead of replacing this field, call `operator.add` to merge the incoming value with the existing value."*

```python
# Normal field — last writer REPLACES:
summaries: list   # worker 1 sets ["A"], worker 2 sets ["B"] → result is ["B"]

# Annotated field — all writers APPEND:
summaries: Annotated[list, operator.add]
# worker 1 sets ["A"], worker 2 sets ["B"] → result is ["A", "B"]
```

### How the Merge Sequence Works

```
Initial state:   summaries = []

Worker 1 returns {"summaries": ["summary of doc A"]}
  LangGraph calls: operator.add([], ["summary of doc A"])
  New state:       summaries = ["summary of doc A"]

Worker 2 returns {"summaries": ["summary of doc B"]}
  LangGraph calls: operator.add(["summary A"], ["summary of doc B"])
  New state:       summaries = ["summary of doc A", "summary of doc B"]

...and so on for all N workers.

aggregate() receives: summaries = ["summary A", "summary B", ...N items]
```

### Reducer Comparison

| Annotation | Reducer | Result for 3 workers each returning `["x"]` |
|------------|---------|--------------------------------------------|
| `list` | Replace (default) | `["x"]` (only last write survives) |
| `Annotated[list, operator.add]` | Append / concat | `["x", "x", "x"]` |
| `Annotated[int, operator.add]` | Sum | `3` (if each returns `1`) |
| Custom lambda | Any merge function | Whatever you define |

In [ ]:
# ─── 3-A: operator.add in action ──────────────────────────────────────────────
# This is the exact reducer LangGraph calls when merging parallel worker outputs.

print("operator.add with lists — the fan-in reducer:\n")

existing = []
worker_outputs = ["summary A", "summary B", "summary C", "summary D"]

for i, summary in enumerate(worker_outputs):
    incoming = [summary]
    existing = operator.add(existing, incoming)
    print(f"  After worker {i + 1} finishes: {existing}")

print()
print("aggregate() will receive:", existing)
print()
print("Key point: ORDER in the merged list depends on which worker finishes first.")
print("Do NOT assume summaries[0] corresponds to documents[0] in parallel execution.")

In [ ]:
# ─── 3-B: Custom reducers — beyond operator.add ───────────────────────────────
# Any binary function (a, b) -> merged can serve as a reducer.

# 1. Sum reducer — useful for counting tokens, scores, etc.
def sum_reducer(a: int, b: int) -> int:
    return a + b

running = 0
for worker_count in [3, 7, 2, 5]:
    running = sum_reducer(running, worker_count)
print(f"Sum reducer (3 + 7 + 2 + 5): {running}")

# 2. Deduplication reducer — useful for keyword / entity extraction
def dedup_reducer(a: list, b: list) -> list:
    seen = set(a)
    return a + [x for x in b if x not in seen]

result: list = []
for batch in [["apple", "banana"], ["banana", "cherry"], ["apple", "date"]]:
    result = dedup_reducer(result, batch)
print(f"Dedup reducer: {result}")

# 3. Dict merge reducer — useful for keyed per-document results
def dict_merge_reducer(a: dict, b: dict) -> dict:
    return {**a, **b}

merged: dict = {}
for kv in [{"doc_0": "summary A"}, {"doc_1": "summary B"}, {"doc_2": "summary C"}]:
    merged = dict_merge_reducer(merged, kv)
print(f"Dict merge reducer: {merged}")

print()
print("Use any of these in Annotated[T, your_reducer] to change fan-in behavior.")

---

## Part 4 — Building the Parallel Graph

---

### The State Schema

The state has two distinct roles living in the same TypedDict:

```python
class State(TypedDict):
    # ── Shared (whole-graph) fields ───────────────────────────────────────────
    documents: list[str]       # input: the full list of docs to process
    final_summary: str         # output: the aggregate result

    # ── Per-worker field (set by each Send) ───────────────────────────────────
    doc: str                   # one document per worker — populated by Send

    # ── Fan-in field (merged across all workers) ──────────────────────────────
    summaries: Annotated[list, operator.add]   # grows as each worker finishes
```

### Node Roles

| Component | Type | Runs | Receives | Returns |
|-----------|------|------|----------|---------|
| `distribute` | Conditional edge function | Once at START | Full graph state | `[Send(...), ...]` |
| `summarize` | Worker node | N times in parallel | Per-item state slice | `{"summaries": [one_item]}` |
| `aggregate` | Fan-in node | Once after all workers | Merged state | `{"final_summary": "..."}` |

In [ ]:
# ─── 4-A: Define the state schema ─────────────────────────────────────────────

class State(TypedDict):
    documents: list[str]                       # shared input
    doc: str                                   # per-worker (set by Send)
    summaries: Annotated[list, operator.add]   # fan-in accumulator
    final_summary: str                         # shared output


print("State fields and their types:\n")
for field, annotation in State.__annotations__.items():
    print(f"  {field:20} {annotation}")

print()
print("'doc' is per-worker — each Send sets it to a different document string.")
print("'summaries' uses Annotated so LangGraph appends (not replaces) each output.")

In [ ]:
# ─── 4-B: Define the three graph components ───────────────────────────────────

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def distribute(state: State) -> list[Send]:
    """Fan-out: return one Send per document.

    This is a conditional edge function, not a node.
    Called once, returns N Send objects.
    LangGraph dispatches all N simultaneously.
    Wall-clock time = slowest single worker, not sum.
    """
    return [
        Send("summarize", {"doc": doc, "summaries": []})
        for doc in state["documents"]
    ]


def summarize(state: State) -> dict:
    """Worker node — runs N times in parallel, one per document.

    state["doc"] contains exactly one document (set by Send).
    Returns a list with exactly one item so operator.add can merge it.
    """
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Summarize this in exactly one sentence:\n\n{doc}")]
    )
    summary = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    return {"summaries": [summary]}


def aggregate(state: State) -> dict:
    """Fan-in node — runs exactly once after ALL workers complete.

    state["summaries"] already contains all N summaries,
    merged by operator.add across all workers.
    """
    bullet_list = "\n".join(f"- {s}" for s in state["summaries"])
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Write a 2-sentence synthesis from these summaries:\n{items}")]
    )
    result = (prompt | llm | StrOutputParser()).invoke({"items": bullet_list})
    return {"final_summary": result}


print("Components defined:")
print("  distribute  — conditional edge, returns [Send, ...]")
print("  summarize   — worker node, runs N times in parallel")
print("  aggregate   — fan-in node, runs once with all summaries")

In [ ]:
# ─── 4-C: Assemble and compile the graph ──────────────────────────────────────

graph = StateGraph(State)

# Add nodes — distribute is NOT a node, it is the edge function
graph.add_node("summarize", summarize)
graph.add_node("aggregate", aggregate)

# Wire the edges:
#   START → distribute() → [Send("summarize", ...), ...]   (fan-out)
#   summarize → aggregate                                   (each worker feeds aggregate)
#   aggregate → END
graph.add_conditional_edges(START, distribute, ["summarize"])  # third arg = allowed targets
graph.add_edge("summarize", "aggregate")
graph.add_edge("aggregate", END)

app = graph.compile()

print("Graph compiled successfully. Topology:")
print()
print("  START")
print("    │ distribute() returns [Send, Send, Send, ...]")
print("    ├── summarize (doc[0]) ─┐")
print("    ├── summarize (doc[1]) ─┤  all run in parallel")
print("    ├── summarize (doc[2]) ─┤")
print("    └── summarize (doc[N]) ─┘")
print("                            │  Annotated[list, add] merges outputs")
print("                        aggregate")
print("                            │")
print("                           END")

---

## Part 5 — Running and Observing the Graph

---

### How to Invoke

The graph expects an initial state dict. Supply every field — even per-worker fields that `Send` will overwrite:

```python
result = app.invoke({
    "documents": [...],    # required: the list to fan out over
    "summaries": [],       # required: start empty, workers will append
    "doc": "",             # required: placeholder — Send overwrites per worker
    "final_summary": "",   # required: placeholder — aggregate sets this
})
# result is the final State dict after the graph completes
```

In [ ]:
# ─── 5-A: Run the parallel map-reduce ─────────────────────────────────────────

SAMPLE_DOCS = [
    "LangGraph 0.6 introduced the Send API, enabling true parallel fan-out across "
    "independent sub-tasks without blocking the main graph thread.",
    "ReAct agents interleave reasoning steps (Thought) with tool calls (Action) and "
    "observations, letting the model decide dynamically which tool to call next.",
    "Retrieval-Augmented Generation (RAG) pipelines combine a dense retriever with a "
    "generative LLM, grounding answers in retrieved documents to reduce hallucination.",
    "Human-in-the-loop checkpointing lets LangGraph agents pause mid-run, surface a "
    "decision to a human, and resume from the exact saved state after approval.",
]

t0 = time.perf_counter()
result = app.invoke({
    "documents": SAMPLE_DOCS,
    "doc": "",
    "summaries": [],
    "final_summary": "",
})
elapsed = time.perf_counter() - t0

print(f"Ran {len(SAMPLE_DOCS)} workers in parallel. Total time: {elapsed:.2f}s")
print()
print(f"Per-document summaries ({len(result['summaries'])} total):")
for i, s in enumerate(result["summaries"], 1):
    print(f"  [{i}] {s}")

print(f"\nFinal synthesis:\n  {result['final_summary']}")

In [ ]:
# ─── 5-B: Does parallel mean deterministic order? ─────────────────────────────
# Run the graph twice and compare the order of summaries.
# Parallel workers finish in a non-deterministic order.

print("Run 1 — summary order:")
run1 = app.invoke({"documents": SAMPLE_DOCS, "doc": "", "summaries": [], "final_summary": ""})
for i, s in enumerate(run1["summaries"]):
    print(f"  [{i}] {s[:70]}...")

print()
print("Run 2 — summary order:")
run2 = app.invoke({"documents": SAMPLE_DOCS, "doc": "", "summaries": [], "final_summary": ""})
for i, s in enumerate(run2["summaries"]):
    print(f"  [{i}] {s[:70]}...")

same_order = run1["summaries"] == run2["summaries"]
print()
print(f"Same order both runs: {same_order}")
print()
print("Lesson: never write logic that relies on summaries[0] mapping to documents[0].")
print("To preserve index mapping, pass doc_index through the Send state (see Part 6).")

In [ ]:
# ─── 5-C: Scale up — fan-out over 8 documents ─────────────────────────────────
# Demonstrates that Send scales with no code changes.

LARGE_DOC_SET = [
    "LangGraph enables stateful multi-actor LLM applications using graph-based workflows.",
    "The Send API dispatches N independent node calls simultaneously from a single edge.",
    "Annotated fields with operator.add merge parallel outputs automatically.",
    "Human-in-the-loop nodes interrupt graph execution for human review and approval.",
    "Checkpointing saves graph state so workflows survive process restarts.",
    "RAG Fusion uses multiple query variants and reciprocal rank fusion to improve recall.",
    "Subgraphs allow composing independent smaller graphs into a master graph.",
    "Streaming in LangGraph surfaces intermediate node outputs to clients in real time.",
]

t0 = time.perf_counter()
large_result = app.invoke({
    "documents": LARGE_DOC_SET,
    "doc": "",
    "summaries": [],
    "final_summary": "",
})
large_elapsed = time.perf_counter() - t0

print(f"Processed {len(LARGE_DOC_SET)} documents in {large_elapsed:.2f}s")
print(f"Summaries collected: {len(large_result['summaries'])}")
print()
for i, s in enumerate(large_result["summaries"], 1):
    print(f"  [{i}] {s[:80]}...")
print(f"\nFinal synthesis:\n  {large_result['final_summary']}")

---

## Part 6 — Variations: Index Preservation and Conditional Routing

---

### Preserving Document-to-Summary Mapping

Because workers finish in non-deterministic order, `summaries[0]` is not guaranteed to correspond to `documents[0]`. The fix: include a `doc_index` in each `Send` state so workers can tag their output.

### Routing to Different Workers

Send can dispatch to *different* node names based on each item's properties. This lets you route short documents to a fast worker and long documents to a thorough one:

```python
def distribute_routed(state):
    return [
        Send("quick_summarize" if len(doc) < 200 else "deep_summarize", {"doc": doc})
        for doc in state["documents"]
    ]
# Wire both workers to the same aggregate node:
graph.add_edge("quick_summarize", "aggregate")
graph.add_edge("deep_summarize", "aggregate")
```

### Use Cases for Parallel Subgraphs

| Use Case | Fan-out over | Workers do | Aggregate does |
|----------|-------------|------------|---------------|
| **Document summarization** | N docs | Summarize each | Synthesize summaries |
| **Multi-query RAG** | N query variants | Retrieve docs for each | RRF merge + deduplicate |
| **Parallel evaluation** | N test cases | Score each | Average / report |
| **Multi-language translation** | N target languages | Translate | Return all translations |
| **Batch classification** | N items | Classify each | Aggregate label counts |

In [ ]:
# ─── 6-A: Preserve document index across parallel workers ─────────────────────

class IndexedState(TypedDict):
    documents: list[str]
    doc: str
    doc_index: int                             # track which doc this worker handles
    summaries: Annotated[list, operator.add]   # list of {index, summary} dicts
    final_summary: str


def distribute_indexed(state: IndexedState) -> list[Send]:
    return [
        Send("summarize_indexed", {"doc": doc, "doc_index": i, "summaries": []})
        for i, doc in enumerate(state["documents"])
    ]


def summarize_indexed(state: IndexedState) -> dict:
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Summarize this in exactly one sentence:\n\n{doc}")]
    )
    summary = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    # Return the index alongside the summary so aggregate can re-sort
    return {"summaries": [{"index": state["doc_index"], "summary": summary}]}


def aggregate_indexed(state: IndexedState) -> dict:
    # Sort by original document index before synthesising
    ordered = sorted(state["summaries"], key=lambda x: x["index"])
    bullet_list = "\n".join(f"- {item['summary']}" for item in ordered)
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Write a 2-sentence synthesis from these summaries:\n{items}")]
    )
    result = (prompt | llm | StrOutputParser()).invoke({"items": bullet_list})
    return {"final_summary": result}


indexed_graph = StateGraph(IndexedState)
indexed_graph.add_node("summarize_indexed", summarize_indexed)
indexed_graph.add_node("aggregate_indexed", aggregate_indexed)
indexed_graph.add_conditional_edges(START, distribute_indexed, ["summarize_indexed"])
indexed_graph.add_edge("summarize_indexed", "aggregate_indexed")
indexed_graph.add_edge("aggregate_indexed", END)
indexed_app = indexed_graph.compile()

indexed_result = indexed_app.invoke({
    "documents": SAMPLE_DOCS,
    "doc": "",
    "doc_index": 0,
    "summaries": [],
    "final_summary": "",
})

print("Summaries sorted by original document index:")
ordered = sorted(indexed_result["summaries"], key=lambda x: x["index"])
for item in ordered:
    print(f"  [doc {item['index']}] {item['summary'][:80]}...")

In [ ]:
# ─── 6-B: Conditional routing to different worker nodes ───────────────────────

class RoutedState(TypedDict):
    documents: list[str]
    doc: str
    summaries: Annotated[list, operator.add]
    final_summary: str


SHORT_THRESHOLD = 100  # characters


def distribute_routed(state: RoutedState) -> list[Send]:
    """Route short docs to quick_summarize, long docs to deep_summarize."""
    sends = []
    for doc in state["documents"]:
        if len(doc) <= SHORT_THRESHOLD:
            sends.append(Send("quick_summarize", {"doc": doc, "summaries": []}))
        else:
            sends.append(Send("deep_summarize", {"doc": doc, "summaries": []}))
    return sends


def quick_summarize(state: RoutedState) -> dict:
    """Fast worker for short documents."""
    prompt = ChatPromptTemplate.from_messages(
        [("human", "In 8 words or fewer, what is this about?\n\n{doc}")]
    )
    s = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    return {"summaries": [f"[quick] {s}"]}


def deep_summarize(state: RoutedState) -> dict:
    """Thorough worker for long documents."""
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Write a detailed one-sentence summary covering the key point:\n\n{doc}")]
    )
    s = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    return {"summaries": [f"[deep] {s}"]}


def aggregate_routed(state: RoutedState) -> dict:
    combined = "\n".join(f"- {s}" for s in state["summaries"])
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Synthesize these mixed summaries into two sentences:\n{items}")]
    )
    return {"final_summary": (prompt | llm | StrOutputParser()).invoke({"items": combined})}


routed_graph = StateGraph(RoutedState)
routed_graph.add_node("quick_summarize", quick_summarize)
routed_graph.add_node("deep_summarize", deep_summarize)
routed_graph.add_node("aggregate_routed", aggregate_routed)
routed_graph.add_conditional_edges(
    START, distribute_routed,
    ["quick_summarize", "deep_summarize"]  # list ALL possible targets
)
# Both workers feed the same aggregate node
routed_graph.add_edge("quick_summarize", "aggregate_routed")
routed_graph.add_edge("deep_summarize", "aggregate_routed")
routed_graph.add_edge("aggregate_routed", END)
routed_app = routed_graph.compile()

mixed_docs = [
    "LangGraph is a framework.",  # short → quick_summarize
    "Short note on RAG.",          # short → quick_summarize
    SAMPLE_DOCS[0],                # long  → deep_summarize
    SAMPLE_DOCS[2],                # long  → deep_summarize
]

print("Routing preview:")
for doc in mixed_docs:
    label = "quick" if len(doc) <= SHORT_THRESHOLD else "deep "
    print(f"  [{label}] ({len(doc):3d} chars) {doc[:60]}")

print()
routed_result = routed_app.invoke(
    {"documents": mixed_docs, "doc": "", "summaries": [], "final_summary": ""}
)
print("Worker outputs:")
for s in routed_result["summaries"]:
    print(f"  {s}")
print(f"\nFinal synthesis:\n  {routed_result['final_summary']}")

---

## Exercises

---

### Exercise 1 — Keyword Extractor

Change the worker task. Instead of summarizing, each worker should extract the **three most important keywords** from its document and return them as a Python list.

The aggregate node should receive a flat list of all keywords from all workers and return a **deduplicated list** of unique keywords across the entire corpus.

**Hint:** Have each worker return keywords as a flat list (three strings, not a list of lists). `operator.add` on `["a", "b", "c"]` + `["d", "e", "f"]` gives `["a", "b", "c", "d", "e", "f"]` — which is exactly what you want.

### Exercise 2 — Scale and Time

Extend `SAMPLE_DOCS` to 10 items. Time both:
- The parallel graph run with `app.invoke()`
- A sequential Python loop calling the LLM directly for each doc

Calculate the actual speedup. Does it match the theoretical max of N? Why might it not?

### Exercise 3 — Conditional Fan-Out by Topic

Before distributing, classify each document as `"technical"` or `"general"` (a simple keyword heuristic is fine). Route technical docs to a worker that formats output as `"CONCEPT: ... | IMPLICATION: ..."`, and general docs to the standard one-sentence summarizer.

The aggregate node should handle the mixed output list and synthesize both types together.

In [ ]:
# ── Exercise 1 Starter — Keyword Extractor ────────────────────────────────────

class KeywordState(TypedDict):
    documents: list[str]
    doc: str
    keywords: Annotated[list, operator.add]  # flat list — each worker adds 3 strings
    top_keywords: list[str]


def distribute_kw(state: KeywordState) -> list[Send]:
    # TODO: return one Send per document targeting "extract_keywords"
    pass


def extract_keywords(state: KeywordState) -> dict:
    # TODO: prompt the LLM to extract 3 keywords from state["doc"]
    # Parse the response into a flat list of 3 strings
    # Return {"keywords": ["kw1", "kw2", "kw3"]}
    pass


def aggregate_keywords(state: KeywordState) -> dict:
    # TODO: deduplicate state["keywords"] while preserving order
    # Return {"top_keywords": [unique keywords]}
    pass


# TODO: build and compile the graph, then invoke it on SAMPLE_DOCS

In [ ]:
# ── Exercise 2 Starter — Scale and Time ───────────────────────────────────────

MY_DOCS = [
    # TODO: add 10 document strings here
    "LangGraph is a framework for building stateful LLM applications.",
    "The Send API enables parallel fan-out in LangGraph.",
    # ... add 8 more ...
]

# Parallel run
t0 = time.perf_counter()
# TODO: invoke app on MY_DOCS
parallel_time_real = time.perf_counter() - t0

# Sequential baseline — same LLM call, one at a time
seq_prompt = ChatPromptTemplate.from_messages(
    [("human", "Summarize this in exactly one sentence:\n\n{doc}")]
)
seq_chain = seq_prompt | llm | StrOutputParser()

t0 = time.perf_counter()
seq_summaries = []
for doc in MY_DOCS:
    # TODO: call seq_chain for each doc
    pass
sequential_time_real = time.perf_counter() - t0

print(f"Documents       : {len(MY_DOCS)}")
print(f"Parallel time   : {parallel_time_real:.2f}s")
print(f"Sequential time : {sequential_time_real:.2f}s")
# TODO: calculate and print actual speedup

In [ ]:
# ── Exercise 3 Starter — Conditional Fan-Out by Topic ─────────────────────────

class TopicState(TypedDict):
    documents: list[str]
    doc: str
    summaries: Annotated[list, operator.add]
    final_summary: str


def classify_doc(doc: str) -> str:
    """Returns 'technical' or 'general'. Replace with LLM call for accuracy."""
    tech_keywords = {"api", "send", "graph", "rag", "node", "langgraph", "model", "llm"}
    return "technical" if any(kw in doc.lower() for kw in tech_keywords) else "general"


def distribute_by_topic(state: TopicState) -> list[Send]:
    # TODO: classify each doc and route to "technical_worker" or "general_worker"
    pass


def technical_worker(state: TopicState) -> dict:
    # TODO: extract "CONCEPT: ... | IMPLICATION: ..."
    pass


def general_worker(state: TopicState) -> dict:
    # TODO: one-sentence summary
    pass


def aggregate_mixed(state: TopicState) -> dict:
    # TODO: synthesize the mixed list of summaries
    pass


# TODO: build and compile the graph
# Remember: both workers must have edges to the aggregate node

---

## What's Next?

You now have a complete foundation in LangGraph parallel execution. Here's where to go deeper:

### Apply the pattern to retrieval
- **Example 26 — RAG Fusion** (`../26-rag-fusion/`): the Send API applied to parallel retrieval — fan out N query variants, retrieve for each, merge with Reciprocal Rank Fusion. A direct extension of what you built here.

### Compose multiple agents
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): multiple specialist agents sharing one state graph with parallel fan-out.

### Evaluate at scale with the same pattern
- **Example 29 — LLM Judge Harness** (`../29-llm-judge-harness/`): use parallel subgraph execution to score N test cases simultaneously — the identical Send API pattern applied to evaluation.

### Go deeper on LangGraph primitives
- **Checkpointing**: `graph.compile(checkpointer=MemorySaver())` — saves state between runs so parallel workflows survive interruption
- **Streaming**: `app.astream()` — surfaces intermediate worker outputs to the client as they complete
- **Subgraphs**: compile a `StateGraph` and use it as a node inside a parent graph — enables hierarchical parallelism

### Further reading
- LangGraph Send API how-to: https://langchain-ai.github.io/langgraph/how-tos/map-reduce/
- Dean & Ghemawat (2004). *MapReduce: Simplified Data Processing on Large Clusters.* OSDI. https://static.googleusercontent.com/media/research.google.com/en//archive/mapreduce-osdi04.pdf
- LangGraph conceptual guide — low-level API: https://langchain-ai.github.io/langgraph/concepts/low_level/#send
- LangGraph subgraphs guide: https://langchain-ai.github.io/langgraph/concepts/subgraphs/

---

*Workshop complete. Next: example 26 (RAG Fusion) to see the Send API applied to retrieval with RRF merging.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 Answer — Keyword Extractor

**Key insight:** Have each worker return keywords as a flat list of strings (not a nested list). `operator.add` on `["a", "b", "c"]` + `["d", "e", "f"]` gives a single flat list `["a", "b", "c", "d", "e", "f"]` — exactly what aggregate needs to deduplicate.

**What to watch for:** If the LLM returns `"kw1, kw2, kw3"` as a single string, split it in the worker before returning. Never return a list of lists — `operator.add([["a"]], [["b"]])` gives `[["a"], ["b"]]`, not `["a", "b"]`.

In [ ]:
# Answer Key — Exercise 1

class KeywordStateSol(TypedDict):
    documents: list[str]
    doc: str
    keywords: Annotated[list, operator.add]  # flat — each worker adds 3 strings
    top_keywords: list[str]


def distribute_kw_sol(state: KeywordStateSol) -> list[Send]:
    return [
        Send("extract_keywords_sol", {"doc": doc, "keywords": []})
        for doc in state["documents"]
    ]


def extract_keywords_sol(state: KeywordStateSol) -> dict:
    prompt = ChatPromptTemplate.from_messages(
        [("human",
          "Extract the 3 most important keywords from this text. "
          "Reply with exactly 3 words separated by commas, nothing else.\n\n{doc}")]
    )
    raw = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    # Parse "keyword1, keyword2, keyword3" → flat list of 3 strings
    kws = [kw.strip().lower() for kw in raw.split(",") if kw.strip()]
    return {"keywords": kws[:3]}  # operator.add will concatenate these with other workers'


def aggregate_keywords_sol(state: KeywordStateSol) -> dict:
    # dict.fromkeys preserves insertion order while deduplicating
    unique = list(dict.fromkeys(state["keywords"]))
    return {"top_keywords": unique}


kw_graph = StateGraph(KeywordStateSol)
kw_graph.add_node("extract_keywords_sol", extract_keywords_sol)
kw_graph.add_node("aggregate_keywords_sol", aggregate_keywords_sol)
kw_graph.add_conditional_edges(START, distribute_kw_sol, ["extract_keywords_sol"])
kw_graph.add_edge("extract_keywords_sol", "aggregate_keywords_sol")
kw_graph.add_edge("aggregate_keywords_sol", END)
kw_app = kw_graph.compile()

kw_result = kw_app.invoke({
    "documents": SAMPLE_DOCS,
    "doc": "",
    "keywords": [],
    "top_keywords": [],
})

print(f"Raw keywords ({len(kw_result['keywords'])} from {len(SAMPLE_DOCS)} workers):")
print(f"  {kw_result['keywords']}")
print(f"\nDeduplicated top keywords ({len(kw_result['top_keywords'])} unique):")
print(f"  {kw_result['top_keywords']}")

### Exercise 2 Answer — Scale and Time

**Expected outcome:** The parallel run should be significantly faster. The speedup is typically 3–6x rather than the theoretical N because:

1. **API rate limits** — OpenAI limits concurrent requests per account tier
2. **Network overhead** — each request has TCP round-trip overhead regardless of parallelism
3. **Token generation time** — longer outputs take longer regardless of parallelism

**What to observe:** Parallel time grows sub-linearly with N (adding 2× more docs adds much less than 2× time), while sequential time grows linearly.

In [ ]:
# Answer Key — Exercise 2

my_docs_sol = [
    "LangGraph is a framework for building stateful LLM applications.",
    "The Send API enables parallel fan-out in LangGraph graphs.",
    "Annotated fields with operator.add merge parallel outputs automatically.",
    "RAG grounds LLM answers in retrieved documents to reduce hallucination.",
    "Human-in-the-loop checkpointing pauses graph execution for human approval.",
    "Vector stores index document embeddings for fast similarity search.",
    "LCEL pipes LangChain components together using the | operator.",
    "LangSmith provides observability for LangChain and LangGraph applications.",
    "Streaming in LangGraph surfaces intermediate node outputs in real time.",
    "Subgraphs allow composing smaller graphs into a larger master graph.",
]

# Parallel
t0 = time.perf_counter()
par_result = app.invoke({
    "documents": my_docs_sol,
    "doc": "",
    "summaries": [],
    "final_summary": "",
})
par_time = time.perf_counter() - t0

# Sequential
seq_chain_sol = (
    ChatPromptTemplate.from_messages(
        [("human", "Summarize this in exactly one sentence:\n\n{doc}")]
    )
    | llm
    | StrOutputParser()
)
t0 = time.perf_counter()
seq_sums = [seq_chain_sol.invoke({"doc": doc}) for doc in my_docs_sol]
seq_time = time.perf_counter() - t0

print(f"Documents       : {len(my_docs_sol)}")
print(f"Parallel time   : {par_time:.2f}s")
print(f"Sequential time : {seq_time:.2f}s")
print(f"Actual speedup  : {seq_time / par_time:.1f}x")
print(f"Theoretical max : {len(my_docs_sol):.0f}x")
print()
print("Speedup < N due to API rate limits, network overhead, and token generation time.")

### Exercise 3 Answer — Conditional Fan-Out by Topic

**Key insight:** When two different worker nodes both feed the same aggregate, you must add an edge from **each** worker to aggregate. A common mistake is wiring only one.

The allowed targets list in `add_conditional_edges` must include **all** possible target node names, or LangGraph will raise an error at runtime when it tries to route to a node not in the list.

In [ ]:
# Answer Key — Exercise 3

class TopicStateSol(TypedDict):
    documents: list[str]
    doc: str
    summaries: Annotated[list, operator.add]
    final_summary: str


def classify_doc_sol(doc: str) -> str:
    tech_kws = {"api", "send", "graph", "rag", "node", "langgraph",
                "model", "llm", "vector", "embedding", "retriev"}
    return "technical" if any(kw in doc.lower() for kw in tech_kws) else "general"


def distribute_by_topic_sol(state: TopicStateSol) -> list[Send]:
    sends = []
    for doc in state["documents"]:
        node = "technical_worker_sol" if classify_doc_sol(doc) == "technical" else "general_worker_sol"
        sends.append(Send(node, {"doc": doc, "summaries": []}))
    return sends


def technical_worker_sol(state: TopicStateSol) -> dict:
    prompt = ChatPromptTemplate.from_messages(
        [("human",
          "For this technical text, identify: (1) the core concept and (2) its key implication. "
          "Format: 'CONCEPT: ... | IMPLICATION: ...'\n\n{doc}")]
    )
    s = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    return {"summaries": [f"[tech] {s}"]}


def general_worker_sol(state: TopicStateSol) -> dict:
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Summarize this in one plain-language sentence:\n\n{doc}")]
    )
    s = (prompt | llm | StrOutputParser()).invoke({"doc": state["doc"]})
    return {"summaries": [f"[general] {s}"]}


def aggregate_mixed_sol(state: TopicStateSol) -> dict:
    combined = "\n".join(f"- {s}" for s in state["summaries"])
    prompt = ChatPromptTemplate.from_messages(
        [("human", "Synthesize these technical and general summaries into two coherent sentences:\n{items}")]
    )
    return {"final_summary": (prompt | llm | StrOutputParser()).invoke({"items": combined})}


topic_graph = StateGraph(TopicStateSol)
topic_graph.add_node("technical_worker_sol", technical_worker_sol)
topic_graph.add_node("general_worker_sol", general_worker_sol)
topic_graph.add_node("aggregate_mixed_sol", aggregate_mixed_sol)
topic_graph.add_conditional_edges(
    START, distribute_by_topic_sol,
    ["technical_worker_sol", "general_worker_sol"]  # ALL possible targets
)
# Critical: add edge from BOTH workers to aggregate
topic_graph.add_edge("technical_worker_sol", "aggregate_mixed_sol")
topic_graph.add_edge("general_worker_sol", "aggregate_mixed_sol")
topic_graph.add_edge("aggregate_mixed_sol", END)
topic_app = topic_graph.compile()

mixed_topic_docs = [
    "LangGraph Send API dispatches parallel node calls.",   # technical
    "Sunsets are beautiful because of light scattering.",  # general
    "RAG retrieves documents to ground LLM answers.",      # technical
    "The economy grew by 3% last quarter.",                # general
]

print("Routing:")
for doc in mixed_topic_docs:
    print(f"  [{classify_doc_sol(doc):9}] {doc[:60]}")

print()
topic_result = topic_app.invoke(
    {"documents": mixed_topic_docs, "doc": "", "summaries": [], "final_summary": ""}
)
print("Worker outputs:")
for s in topic_result["summaries"]:
    print(f"  {s[:90]}")
print(f"\nFinal synthesis:\n  {topic_result['final_summary']}")